# LLaMA-7B — QK Block Importance — FINAL RUN-ALL SWEEP

Fresh-runtime notebook for the **70% → 80% → 90%** QK block-importance sweep.

**Run All behavior:** setup/model/tokenizer/LongBench → estimator → attention patch → 70% → 80% → 90%.

The completed Dense and 50% results are embedded as official reference values and are **not rerun**. Diagnostic/test cells are intentionally excluded from the Run All path.


In [ ]:
# -*- coding: utf-8 -*-
"""OFFICIAL_SPARSITY_TEST.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1qWVBItB1Ld2o9lnK4-lbgjbaWPGDtur-
"""

!pip install -q transformers accelerate bitsandbytes sentencepiece datasets==2.21.0 rouge_score

import torch
import math
import types
import gc
import re
import string
import pandas as pd

from datasets import load_dataset
from transformers.models.llama.modeling_llama import apply_rotary_pos_emb

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

MODEL_NAME = "meta-llama/Llama-2-7b-chat-hf"

ALPHA = 20
SEQ_LEN = 2048

print(torch.cuda.get_device_name(0))

# Clone LongBench repository

import os

if not os.path.exists("/content/LongBench"):
    !git clone https://github.com/THUDM/LongBench.git /content/LongBench
else:
    print("LongBench already exists")

from huggingface_hub import login

login()

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    llm_int8_enable_fp32_cpu_offload=True
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)


model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager"
)

model.eval()

from datasets import load_dataset


tasks = [
    "narrativeqa",
    "qasper",
    "gov_report"
]

SAMPLES_PER_TASK = 15
MAX_TOTAL_SAMPLES = 50

samples = []

for task in tasks:

    print("="*50)
    print("Loading:", task)

    ds = load_dataset(
        "THUDM/LongBench",
        task,
        split="test"
    )

    print(ds)
    print("Fields:", ds.column_names)

    for item in ds.select(range(min(SAMPLES_PER_TASK, len(ds)))):
        samples.append({
            "task": task,
            "context": item["context"],
            "input": item["input"],
            "answers": item["answers"],
            "all_classes": item.get("all_classes", []),
            "length": item.get("length", None)
        })

samples = samples[:MAX_TOTAL_SAMPLES]

print("="*50)
print("TOTAL SAMPLES:", len(samples))
from collections import Counter
print("BY TASK:", Counter(s["task"] for s in samples))
print(samples[0])

from transformers.models.llama.modeling_llama import LlamaAttention

# restore toàn bộ attention forward về class gốc
for layer in model.model.layers:

    layer.self_attn.forward = (
        LlamaAttention.forward.__get__(
            layer.self_attn,
            type(layer.self_attn)
        )
    )

print(model.model.layers[0].self_attn.forward)


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 9.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2024.6.1 which is incompatible.
Tesla T4
Cloning into '/content/LongBench'...
remote: Enumerating objects: 312, done.
remote: Counting objects: 100% (94/94), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 312 (delta 74), reused 66 (delta 66), pack-reused 218 (from 1)
Receiving objects: 100% (312/

config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

Dataset({
    features: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id'],
    num_rows: 200
})
Fields: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id']
Loading: qasper


Generating test split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id'],
    num_rows: 200
})
Fields: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id']
Loading: gov_report


Generating test split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id'],
    num_rows: 200
})
Fields: ['input', 'context', 'answers', 'length', 'dataset', 'language', 'all_classes', '_id']
TOTAL SAMPLES: 45
BY TASK: Counter({'narrativeqa': 15, 'qasper': 15, 'gov_report': 15})
{'task': 'narrativeqa', 'context': 'Transcribed from the 1915 Martin Secker edition by David Price, email\nccx074@pglaf.org\n\n                          [Picture: Book cover]\n\n\n\n\n\n                                   THE\n                                COXON FUND\n\n\n                              BY HENRY JAMES\n\n                      [Picture: Decorative graphic]\n\n                                * * * * *\n\n                          LONDON: MARTIN SECKER\n                     NUMBER FIVE JOHN STREET ADELPHI\n\n                                * * * * *\n\n                    This edition first published 1915\n\n                       The text follows that of the\n  

In [ ]:
# ============================================================
# QK BLOCK IMPORTANCE ESTIMATOR — FIXED v2
# ============================================================
#
# 32x32 block importance for QK attention.
#
# IMPORTANT:
# - Causal-invalid blocks are NOT pruning candidates.
# - Only blocks with at least one finite QK value AND
#   positive L1 importance participate in Top-K.
# - Partially valid causal blocks are still candidates.
# - API is unchanged:
#       get_qk_importance_mask(...)
#
# Compatible with the existing attention patch.
# ============================================================


BLOCK_ROWS = 32
BLOCK_COLS = 32

QK_IMPORTANCE_MODE = "l1"

# First experiment
QK_TARGET_SPARSITY = 50.0


# ============================================================
# Statistics
# ============================================================

qk_stats = {
    "sum_sparsity": 0.0,
    "count": 0,

    "sum_threshold": 0.0,
    "threshold_count": 0,

    "candidate_blocks": 0,
    "kept_blocks": 0,

    # Additional diagnostics
    "zero_importance_blocks": 0,
    "positive_importance_blocks": 0,
}


# ============================================================
# L1 QK BLOCK IMPORTANCE
# ============================================================

def get_qk_importance_mask(
    scores,
    block_rows=BLOCK_ROWS,
    block_cols=BLOCK_COLS,
    target_sparsity=None,
):
    if target_sparsity is None:
        target_sparsity = QK_TARGET_SPARSITY
    """
    Compute a 2-D block keep mask for QK attention.

    For each block B:

        I(B) = mean(|QK_ij|)

    using ONLY finite QK entries.

    Causal-masked entries (-inf) are ignored.

    IMPORTANT:
    Blocks with zero L1 importance are NOT considered
    pruning candidates. This prevents causal-invalid blocks
    from consuming the Top-K budget.

    Partially valid causal blocks remain candidates.

    Target sparsity is applied ONLY to positive-importance
    candidate blocks.

    Returns
    -------
    keep_mask : torch.BoolTensor
        Shape:
            [num_block_rows, num_block_cols]
    """

    rows, cols = scores.shape

    # --------------------------------------------------------
    # Number of complete blocks
    # --------------------------------------------------------

    num_block_rows = rows // block_rows
    num_block_cols = cols // block_cols

    if num_block_rows == 0 or num_block_cols == 0:

        return torch.zeros(
            num_block_rows,
            num_block_cols,
            device=scores.device,
            dtype=torch.bool,
        )

    # --------------------------------------------------------
    # Crop to complete blocks
    # --------------------------------------------------------

    S = scores[
        :num_block_rows * block_rows,
        :num_block_cols * block_cols,
    ]

    # --------------------------------------------------------
    # Reshape into block grid
    #
    # [block_row,
    #  row_inside_block,
    #  block_col,
    #  col_inside_block]
    # --------------------------------------------------------

    blocks = S.reshape(
        num_block_rows,
        block_rows,
        num_block_cols,
        block_cols,
    )

    # --------------------------------------------------------
    # Valid QK entries
    #
    # Causal masking normally produces -inf.
    # --------------------------------------------------------

    valid = torch.isfinite(blocks)

    valid_count = valid.sum(
        dim=(1, 3)
    )

    # Block contains at least one finite QK value.
    has_valid = valid_count > 0

    # --------------------------------------------------------
    # L1 importance
    #
    # I(B) = mean(|QK|)
    #
    # Only finite values contribute.
    # --------------------------------------------------------

    abs_values = torch.where(
        valid,
        blocks.abs(),
        torch.zeros_like(blocks),
    )

    importance = (
        abs_values.sum(dim=(1, 3))
        /
        valid_count.clamp_min(1)
    )

    importance = torch.nan_to_num(
        importance.float(),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    )

    # --------------------------------------------------------
    # Flatten block grid
    # --------------------------------------------------------

    importance_flat = importance.flatten()
    valid_flat = has_valid.flatten()

    # --------------------------------------------------------
    # IMPORTANT FIX
    #
    # Only POSITIVE-importance blocks are candidates.
    #
    # This excludes blocks whose effective QK importance
    # is zero because they contain no meaningful finite
    # attention scores.
    #
    # Partially valid causal blocks remain candidates.
    # --------------------------------------------------------

    positive_flat = importance_flat > 0

    candidate_mask = (
        valid_flat
        & positive_flat
    )

    candidate_idx = torch.nonzero(
        candidate_mask,
        as_tuple=False,
    ).flatten()

    total_candidates = candidate_idx.numel()

    # --------------------------------------------------------
    # Diagnostics
    # --------------------------------------------------------

    zero_importance_count = (
        valid_flat
        & (importance_flat == 0)
    ).sum().item()

    positive_importance_count = (
        valid_flat
        & positive_flat
    ).sum().item()

    qk_stats["zero_importance_blocks"] += (
        zero_importance_count
    )

    qk_stats["positive_importance_blocks"] += (
        positive_importance_count
    )

    # --------------------------------------------------------
    # No positive candidate
    # --------------------------------------------------------

    if total_candidates == 0:

        return torch.zeros(
            num_block_rows * num_block_cols,
            device=scores.device,
            dtype=torch.bool,
        ).reshape(
            num_block_rows,
            num_block_cols,
        )

    # --------------------------------------------------------
    # Determine target sparsity
    # --------------------------------------------------------

    target = max(
        0.0,
        min(
            99.999,
            float(target_sparsity),
        ),
    )

    # --------------------------------------------------------
    # Number of blocks to KEEP
    #
    # Sparsity is now defined over POSITIVE-IMPORTANCE
    # candidate blocks only.
    # --------------------------------------------------------

    keep_count = int(
        round(
            total_candidates
            * (1.0 - target / 100.0)
        )
    )

    # Keep at least one positive candidate.
    keep_count = max(
        1,
        min(
            total_candidates,
            keep_count,
        ),
    )

    # --------------------------------------------------------
    # Candidate importance
    # --------------------------------------------------------

    candidate_importance = (
        importance_flat[candidate_idx]
    )

    # --------------------------------------------------------
    # Top-K important blocks
    # --------------------------------------------------------

    top_values, top_positions = torch.topk(
        candidate_importance,
        k=keep_count,
        largest=True,
        sorted=False,
    )

    selected_idx = (
        candidate_idx[top_positions]
    )

    # --------------------------------------------------------
    # KEEP / DROP mask
    #
    # All positive candidate blocks start as DROP.
    # Top-K are restored to KEEP.
    #
    # Causal-invalid blocks remain False.
    # --------------------------------------------------------

    keep_mask = torch.zeros(
        num_block_rows * num_block_cols,
        device=scores.device,
        dtype=torch.bool,
    )

    keep_mask[selected_idx] = True

    keep_mask = keep_mask.reshape(
        num_block_rows,
        num_block_cols,
    )

    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    qk_stats["candidate_blocks"] += (
        total_candidates
    )

    qk_stats["kept_blocks"] += (
        keep_count
    )

    # --------------------------------------------------------
    # Threshold
    #
    # Smallest positive importance among kept blocks.
    # --------------------------------------------------------

    threshold = top_values.min().item()

    qk_stats["sum_threshold"] += threshold

    qk_stats["threshold_count"] += 1

    return keep_mask


# ============================================================
# STATUS
# ============================================================

print(
    "QK Block Importance estimator loaded."
)

print(
    f"Block size       : "
    f"{BLOCK_ROWS}x{BLOCK_COLS}"
)

print(
    f"Importance mode   : "
    f"{QK_IMPORTANCE_MODE}"
)

print(
    f"Target sparsity   : "
    f"{QK_TARGET_SPARSITY}%"
)

print(
    "Candidate policy  : "
    "positive L1 importance only"
)

print(
    "Causal-zero blocks: "
    "excluded from Top-K"
)

QK Block Importance estimator loaded.
Block size       : 32x32
Importance mode   : l1
Target sparsity   : 50.0%
Candidate policy  : positive L1 importance only
Causal-zero blocks: excluded from Top-K


In [ ]:
# ============================================================
# CHECK ATTENTION PATCH -> ESTIMATOR REFERENCE
# ============================================================

import inspect

print("=" * 70)
print("ATTENTION PATCH / ESTIMATOR REFERENCE CHECK")
print("=" * 70)

print()
print("CURRENT GLOBALS")

print(
    "QK_TARGET_SPARSITY =",
    QK_TARGET_SPARSITY
)

print(
    "BLOCK_ROWS =",
    BLOCK_ROWS
)

print(
    "BLOCK_COLS =",
    BLOCK_COLS
)

print()
print("CURRENT ESTIMATOR")

print(
    "Function object:",
    get_qk_importance_mask
)

print(
    "Function module:",
    get_qk_importance_mask.__module__
)

print(
    "Function name:",
    get_qk_importance_mask.__name__
)

print()
print("ESTIMATOR SOURCE")

try:

    source = inspect.getsource(
        get_qk_importance_mask
    )

    print(
        source[:2000]
    )

except Exception as e:

    print(
        "Could not inspect source:",
        e
    )


# ============================================================
# CHECK DEFAULT ARGUMENT
# ============================================================

print()
print("DEFAULT ARGUMENTS")

print(
    get_qk_importance_mask.__defaults__
)


# ============================================================
# CHECK ATTENTION FUNCTION
# ============================================================

print()
print("ATTENTION FUNCTION")

print(
    "qk_block_importance_attention:",
    qk_block_importance_attention
)

try:

    attention_source = inspect.getsource(
        qk_block_importance_attention
    )

    # Find the estimator call
    idx = attention_source.find(
        "get_qk_importance_mask"
    )

    if idx >= 0:

        print(
            attention_source[
                max(0, idx - 500):
                idx + 1000
            ]
        )

    else:

        print(
            "Estimator call not found "
            "in attention source."
        )

except Exception as e:

    print(
        "Could not inspect attention source:",
        e
    )


# ============================================================
# FINAL
# ============================================================

print()
print("=" * 70)
print("END REFERENCE CHECK")
print("=" * 70)

ATTENTION PATCH / ESTIMATOR REFERENCE CHECK

CURRENT GLOBALS
QK_TARGET_SPARSITY = 90.0
BLOCK_ROWS = 32
BLOCK_COLS = 32

CURRENT ESTIMATOR
Function object: <function get_qk_importance_mask at 0x7b561bece0c0>
Function module: __main__
Function name: get_qk_importance_mask

ESTIMATOR SOURCE
def get_qk_importance_mask(
    scores,
    block_rows=BLOCK_ROWS,
    block_cols=BLOCK_COLS,
    target_sparsity=None,
):
    if target_sparsity is None:
        target_sparsity = QK_TARGET_SPARSITY
    """
    Compute a 2-D block keep mask for QK attention.

    For each block B:

        I(B) = mean(|QK_ij|)

    using ONLY finite QK entries.

    Causal-masked entries (-inf) are ignored.

    IMPORTANT:
    Blocks with zero L1 importance are NOT considered
    pruning candidates. This prevents causal-invalid blocks
    from consuming the Top-K budget.

    Partially valid causal blocks remain candidates.

    Target sparsity is applied ONLY to positive-importance
    candidate blocks.

    Returns


In [ ]:
# ============================================================
# QK BLOCK IMPORTANCE — ATTENTION PATCH
# ============================================================
#
# FINAL CLEAN VERSION
#
# Flow:
#
#   RAW QK
#      ↓
#   L1 block importance estimator
#      ↓
#   keep_mask
#      ↓
#   block pruning
#      ↓
#   original attention mask
#      ↓
#   safety against all -inf rows
#      ↓
#   softmax
#      ↓
#   attention × V
#
# This cell:
#   - does NOT modify the estimator
#   - does NOT reload the model
#   - does NOT run LongBench
#   - does NOT use qk_stats to decide pruning
# ============================================================

import math
import types

import torch
import torch.nn.functional as F


# ============================================================
# GQA HELPER
# ============================================================

def _repeat_kv(hidden_states, n_rep):

    batch, num_kv_heads, seq_len, head_dim = (
        hidden_states.shape
    )

    if n_rep == 1:
        return hidden_states

    hidden_states = hidden_states[:, :, None, :, :]

    hidden_states = hidden_states.expand(
        batch,
        num_kv_heads,
        n_rep,
        seq_len,
        head_dim,
    )

    return hidden_states.reshape(
        batch,
        num_kv_heads * n_rep,
        seq_len,
        head_dim,
    )


# ============================================================
# ATTENTION FORWARD
# ============================================================

def _qk_block_attention_forward(
    self,
    hidden_states,
    position_embeddings=None,
    attention_mask=None,
    past_key_values=None,
    cache_position=None,
    position_ids=None,
    **kwargs,
):

    bsz, q_len, _ = hidden_states.shape

    # --------------------------------------------------------
    # Configuration
    # --------------------------------------------------------

    config = self.config

    num_heads = config.num_attention_heads
    num_kv_heads = config.num_key_value_heads

    head_dim = getattr(
        config,
        "head_dim",
        config.hidden_size // num_heads,
    )

    num_kv_groups = (
        num_heads // num_kv_heads
    )

    # --------------------------------------------------------
    # Q / K / V projections
    # --------------------------------------------------------

    query_states = self.q_proj(
        hidden_states
    )

    key_states = self.k_proj(
        hidden_states
    )

    value_states = self.v_proj(
        hidden_states
    )

    # --------------------------------------------------------
    # Reshape
    # --------------------------------------------------------

    query_states = query_states.view(
        bsz,
        q_len,
        num_heads,
        head_dim,
    ).transpose(1, 2)

    key_states = key_states.view(
        bsz,
        q_len,
        num_kv_heads,
        head_dim,
    ).transpose(1, 2)

    value_states = value_states.view(
        bsz,
        q_len,
        num_kv_heads,
        head_dim,
    ).transpose(1, 2)

    # ========================================================
    # ROPE
    # ========================================================

    if position_embeddings is not None:

        cos, sin = position_embeddings

        def rotate_half(x):

            x1 = x[
                ...,
                : x.shape[-1] // 2
            ]

            x2 = x[
                ...,
                x.shape[-1] // 2 :
            ]

            return torch.cat(
                (-x2, x1),
                dim=-1,
            )

        query_states = (
            query_states * cos
            +
            rotate_half(query_states) * sin
        )

        key_states = (
            key_states * cos
            +
            rotate_half(key_states) * sin
        )

    else:

        try:

            cos, sin = self.rotary_emb(
                value_states,
                position_ids,
            )

            def rotate_half(x):

                x1 = x[
                    ...,
                    : x.shape[-1] // 2
                ]

                x2 = x[
                    ...,
                    x.shape[-1] // 2 :
                ]

                return torch.cat(
                    (-x2, x1),
                    dim=-1,
                )

            query_states = (
                query_states * cos
                +
                rotate_half(query_states) * sin
            )

            key_states = (
                key_states * cos
                +
                rotate_half(key_states) * sin
            )

        except Exception:
            pass

    # ========================================================
    # GQA
    # ========================================================

    key_states = _repeat_kv(
        key_states,
        num_kv_groups,
    )

    value_states = _repeat_kv(
        value_states,
        num_kv_groups,
    )

    # ========================================================
    # RAW QK
    #
    # IMPORTANT:
    # No causal / attention mask is applied here.
    # ========================================================

    raw_scores = torch.matmul(
        query_states,
        key_states.transpose(-2, -1),
    )

    scaling = head_dim ** -0.5

    raw_scores = (
        raw_scores * scaling
    )

    # ========================================================
    # QK BLOCK IMPORTANCE
    #
    # Estimator receives RAW QK.
    # ========================================================

    masked_scores = raw_scores.clone()

    batch_size = raw_scores.shape[0]
    head_count = raw_scores.shape[1]

    for b in range(batch_size):

        for h in range(head_count):

            # ------------------------------------------------
            # Raw QK matrix
            # ------------------------------------------------

            qk = raw_scores[b, h]

            # ------------------------------------------------
            # Direct estimator call
            # ------------------------------------------------

            keep_mask = get_qk_importance_mask(
                qk,
                block_rows=BLOCK_ROWS,
                block_cols=BLOCK_COLS,
                target_sparsity=QK_TARGET_SPARSITY,
            )

            # ------------------------------------------------
            # Convert block mask -> element mask
            # ------------------------------------------------

            element_mask = (
                keep_mask
                .repeat_interleave(
                    BLOCK_ROWS,
                    dim=0,
                )
                .repeat_interleave(
                    BLOCK_COLS,
                    dim=1,
                )
            )

            q_valid = min(
                element_mask.shape[0],
                qk.shape[0],
            )

            k_valid = min(
                element_mask.shape[1],
                qk.shape[1],
            )

            if q_valid == 0 or k_valid == 0:
                continue

            block_region = (
                element_mask[
                    :q_valid,
                    :k_valid,
                ]
            )

            # ------------------------------------------------
            # Apply block pruning to RAW QK
            # ------------------------------------------------

            current_scores = masked_scores[
                b,
                h,
                :q_valid,
                :k_valid,
            ]

            masked_scores[
                b,
                h,
                :q_valid,
                :k_valid,
            ] = torch.where(
                block_region,
                current_scores,
                torch.full_like(
                    current_scores,
                    float("-inf"),
                ),
            )

    # ========================================================
    # ORIGINAL ATTENTION MASK
    #
    # Apply causal/external mask AFTER block pruning.
    # ========================================================

    if attention_mask is not None:

        masked_scores = (
            masked_scores
            + attention_mask
        )

    # ========================================================
    # SAFETY AGAINST ALL -INF ROWS
    #
    # A row containing only -inf causes:
    #
    #     softmax([-inf, -inf, ...]) -> NaN
    #
    # First try to restore the ORIGINAL attention row.
    # ========================================================

    finite_rows = torch.isfinite(
        masked_scores
    ).any(dim=-1)

    invalid_rows = ~finite_rows

    if invalid_rows.any():

        # ----------------------------------------------------
        # Reconstruct original attention scores
        # ----------------------------------------------------

        fallback_scores = raw_scores.clone()

        if attention_mask is not None:

            fallback_scores = (
                fallback_scores
                + attention_mask
            )

        # ----------------------------------------------------
        # Check which fallback rows are valid
        # ----------------------------------------------------

        fallback_valid = torch.isfinite(
            fallback_scores
        ).any(dim=-1)

        restore_rows = (
            invalid_rows
            &
            fallback_valid
        )

        masked_scores = torch.where(
            restore_rows.unsqueeze(-1),
            fallback_scores,
            masked_scores,
        )

    # ========================================================
    # FINAL SAFETY
    #
    # If a row is STILL completely -inf, restore one valid
    # self-position whenever it exists.
    #
    # For the 64-token diagnostic:
    #
    #     query q -> key q
    #
    # is always a valid causal position.
    # ========================================================

    final_valid = torch.isfinite(
        masked_scores
    ).any(dim=-1)

    still_invalid = ~final_valid

    if still_invalid.any():

        invalid_indices = torch.nonzero(
            still_invalid,
            as_tuple=False,
        )

        for idx in invalid_indices:

            b = int(idx[0].item())
            h = int(idx[1].item())
            q = int(idx[2].item())

            if q < masked_scores.shape[-1]:

                masked_scores[
                    b,
                    h,
                    q,
                    q,
                ] = raw_scores[
                    b,
                    h,
                    q,
                    q,
                ]

    # ========================================================
    # FINAL NaN PROTECTION
    #
    # This should normally be unnecessary after the checks
    # above, but prevents invalid softmax input.
    # ========================================================

    masked_scores = torch.nan_to_num(
        masked_scores,
        nan=0.0,
        posinf=0.0,
        neginf=-torch.finfo(
            masked_scores.dtype
        ).max,
    )

    # ========================================================
    # SOFTMAX
    # ========================================================

    attn_weights = F.softmax(
        masked_scores,
        dim=-1,
        dtype=torch.float32,
    ).to(
        query_states.dtype
    )

    # ========================================================
    # ATTENTION × V
    # ========================================================

    attn_output = torch.matmul(
        attn_weights,
        value_states,
    )

    # ========================================================
    # MERGE HEADS
    # ========================================================

    attn_output = (
        attn_output
        .transpose(1, 2)
        .contiguous()
    )

    attn_output = attn_output.reshape(
        bsz,
        q_len,
        num_heads * head_dim,
    )

    # ========================================================
    # OUTPUT PROJECTION
    # ========================================================

    attn_output = self.o_proj(
        attn_output
    )

    return (
        attn_output,
        None,
    )


# ============================================================
# INSTALL PATCH
# ============================================================

patched_layers = 0

for layer in model.model.layers:

    layer.self_attn.forward = types.MethodType(
        _qk_block_attention_forward,
        layer.self_attn,
    )

    patched_layers += 1


# ============================================================
# FORCE EAGER ATTENTION
# ============================================================

try:

    model.config._attn_implementation = "eager"

except Exception:
    pass


# ============================================================
# RESET STATISTICS
# ============================================================

qk_stats["candidate_blocks"] = 0
qk_stats["kept_blocks"] = 0
qk_stats["sum_threshold"] = 0.0
qk_stats["threshold_count"] = 0


# ============================================================
# CONFIRMATION
# ============================================================

print("=" * 60)
print(
    "QK BLOCK IMPORTANCE — FINAL CLEAN ATTENTION PATCH"
)
print("=" * 60)

print(
    "Model                 :",
    type(model).__name__,
)

print(
    "Patched layers        :",
    patched_layers,
)

print(
    "Attention backend     :",
    getattr(
        model.config,
        "_attn_implementation",
        "unknown",
    ),
)

print(
    "Block size            :",
    f"{BLOCK_ROWS}x{BLOCK_COLS}",
)

print(
    "Target sparsity       :",
    QK_TARGET_SPARSITY,
    "%",
)

print(
    "Importance mode       :",
    QK_IMPORTANCE_MODE,
)

print(
    "Estimator input       :",
    "RAW QK BEFORE MASK",
)

print(
    "Block pruning         :",
    "ENABLED",
)

print(
    "Causal mask           :",
    "APPLIED AFTER PRUNING",
)

print(
    "All-inf row protection:",
    "ENABLED",
)

print(
    "qk_stats controls mask:",
    "NO",
)

print("=" * 60)
print(
    "PATCH INSTALLED"
)

print(
    "Next: run ONLY the 64-token small inference test."
)

print("=" * 60)

QK BLOCK IMPORTANCE — FINAL CLEAN ATTENTION PATCH
Model                 : LlamaForCausalLM
Patched layers        : 32
Attention backend     : eager
Block size            : 32x32
Target sparsity       : 90.0 %
Importance mode       : l1
Estimator input       : RAW QK BEFORE MASK
Block pruning         : ENABLED
Causal mask           : APPLIED AFTER PRUNING
All-inf row protection: ENABLED
qk_stats controls mask: NO
PATCH INSTALLED
Next: run ONLY the 64-token small inference test.


In [ ]:
# ============================================================
# LONGBENCH GENERATION / EVALUATION SUPPORT
# RUN-ALL SETUP — NO DENSE EXPERIMENT
# ============================================================

import os
import json
import subprocess
from collections import defaultdict

# Required evaluation packages.
import sys
!{sys.executable} -m pip install -q fuzzywuzzy python-Levenshtein rouge

os.makedirs("pred", exist_ok=True)
os.makedirs("results", exist_ok=True)

CONFIG_DIR = "/content/LongBench/LongBench/config"

prompt_path = f"{CONFIG_DIR}/dataset2prompt.json"
if not os.path.exists(prompt_path):
    raise RuntimeError(
        f"LongBench prompt config not found: {prompt_path}"
    )

with open(prompt_path, "r", encoding="utf-8") as f:
    dataset2prompt = json.load(f)

maxlen_path = f"{CONFIG_DIR}/dataset2maxlen.json"

if os.path.exists(maxlen_path):
    with open(maxlen_path, "r", encoding="utf-8") as f:
        dataset2maxlen = json.load(f)
else:
    dataset2maxlen = {}

print("=" * 60)
print("LONGBENCH SUPPORT READY")
print("=" * 60)
print("Prompt config :", prompt_path)
print("Tasks         :", list(dataset2prompt.keys())[:10])
print("Dense run     : SKIPPED")
print("50% run       : SKIPPED")


def run_longbench_eval(model_name):
    """
    Run the existing LongBench evaluator on pred/<model_name>.
    """
    cmd = [
        sys.executable,
        "/content/LongBench/LongBench/eval.py",
        "--model",
        model_name,
    ]

    result = subprocess.run(
        cmd,
        capture_output=True,
        text=True,
    )

    print(result.stdout)

    if result.stderr:
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(
            f"LongBench evaluation failed for '{model_name}' "
            f"with return code {result.returncode}."
        )

    return result.stdout


def generate_longbench_answer(sample):
    task = sample["task"]

    if task not in dataset2prompt:
        raise KeyError(
            f"Task '{task}' is missing from dataset2prompt.json"
        )

    prompt = dataset2prompt[task].format(**sample)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=SEQ_LEN,
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    max_new_tokens = 64

    if task in dataset2maxlen:
        max_new_tokens = int(dataset2maxlen[task])

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    return tokenizer.decode(
        output[0],
        skip_special_tokens=True,
    )


def export_longbench_prediction(mode, output_name):
    """
    Generate LongBench predictions using the CURRENT model state.

    mode='qk' is the only mode used in this final sweep.
    The already-installed QK attention patch remains active.
    """
    if mode != "qk":
        raise ValueError(
            "FINAL RUN-ALL notebook only supports mode='qk'. "
            "Dense and 50% experiments are intentionally skipped."
        )

    if "qk_block_importance_attention" not in globals():
        raise RuntimeError(
            "QK attention patch function is missing."
        )

    records = []

    print("=" * 60)
    print(f"GENERATING LONG BENCH PREDICTIONS: {output_name}")
    print("=" * 60)

    for idx, sample in enumerate(samples):
        print(
            f"[{idx + 1}/{len(samples)}] "
            f"qk {sample['task']}"
        )

        pred = generate_longbench_answer(sample)

        records.append({
            "pred": pred,
            "answers": sample["answers"],
            "task": sample["task"],
            "all_classes": sample.get("all_classes", []),
            "length": sample.get("length", None),
        })

    save_dir = os.path.join("pred", output_name)
    os.makedirs(save_dir, exist_ok=True)

    grouped_records = defaultdict(list)

    for item in records:
        grouped_records[item["task"]].append(item)

    for task, task_records in grouped_records.items():
        filename = os.path.join(
            save_dir,
            f"{task}.jsonl",
        )

        with open(filename, "w", encoding="utf-8") as f:
            for item in task_records:
                f.write(
                    json.dumps(
                        item,
                        ensure_ascii=False,
                    ) + "\n"
                )

        print("Saved:", filename)

    return save_dir


print()
print("LongBench generation/evaluation functions are ready.")
print("Only QK 70% / 80% / 90% will be executed.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.4/157.4 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 53.0 MB/s eta 0:00:00
LONGBENCH SUPPORT READY
Prompt config : /content/LongBench/LongBench/config/dataset2prompt.json
Tasks         : ['narrativeqa', 'qasper', 'multifieldqa_en', 'multifieldqa_zh', 'hotpotqa', '2wikimqa', 'musique', 'dureader', 'gov_report', 'qmsum']
Dense run     : SKIPPED
50% run       : SKIPPED

LongBench generation/evaluation functions are ready.
Only QK 70% / 80% / 90% will be executed.


In [ ]:
# ============================================================
# QK BLOCK IMPORTANCE — FINAL 70% / 80% / 90% SWEEP
# ============================================================
#
# THIS IS THE ONLY EXPERIMENT CELL IN THE RUN-ALL PATH.
#
# It does NOT:
#   - run Dense baseline
#   - rerun 50%
#   - run threshold diagnostics
#   - change the estimator
#   - change the attention patch
#
# It ONLY runs:
#   70% -> 80% -> 90%
#
# Official completed 50% results are preserved below.
# ============================================================

import json
import math

# ------------------------------------------------------------
# OFFICIAL DENSE RESULT — ALREADY COMPLETED
# ------------------------------------------------------------

OFFICIAL_DENSE_RESULT = {
    "narrativeqa": 0.42,
    "qasper": 1.08,
    "gov_report": 34.17,
}

OFFICIAL_DENSE_AVERAGE = 11.89

# ------------------------------------------------------------
# OFFICIAL 50% RESULT — DO NOT RERUN
# ------------------------------------------------------------

OFFICIAL_50_RESULT = {
    "target_sparsity": 50.0,
    "measured_sparsity": 49.99474892352932,
    "candidate_blocks": 4265792,
    "kept_blocks": 2133120,
    "threshold_count": 2880,
    "average_threshold": 1.816662343343099,
    "dense_average": 11.89,
    "qk_average": 12.13,
    "score_loss": -0.2400000000000002,
    "dense_per_task": OFFICIAL_DENSE_RESULT,
    "qk_per_task": {
        "narrativeqa": 0.44,
        "qasper": 1.12,
        "gov_report": 34.83,
    },
}

SWEEP_LEVELS = [70.0, 80.0, 90.0]

# ------------------------------------------------------------
# SAFETY CHECKS
# ------------------------------------------------------------

required = [
    "model",
    "tokenizer",
    "samples",
    "qk_stats",
    "get_qk_importance_mask",
    "qk_block_importance_attention",
    "export_longbench_prediction",
    "run_longbench_eval",
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "RUN-ALL setup is incomplete. Missing:\n"
        + "\n".join(f"  - {name}" for name in missing)
    )

# These are reference values only. No files from the old account/runtime
# are needed.
dense_result = OFFICIAL_DENSE_RESULT.copy()
dense_avg = OFFICIAL_DENSE_AVERAGE

sweep_results = []

print("=" * 60)
print("FINAL QK BLOCK IMPORTANCE SWEEP")
print("=" * 60)
print("Dense average :", dense_avg)
print("50% QK average:", OFFICIAL_50_RESULT["qk_average"])
print("50% sparsity  :", OFFICIAL_50_RESULT["measured_sparsity"], "%")
print()
print("RUNNING ONLY:", " -> ".join(f"{x:.0f}%" for x in SWEEP_LEVELS))
print("Dense baseline : SKIPPED")
print("50% experiment : SKIPPED")
print("Diagnostics    : SKIPPED")

for sparsity in SWEEP_LEVELS:

    print("\n" + "=" * 60)
    print(f"STARTING QK EXPERIMENT — {sparsity:.0f}% SPARSITY")
    print("=" * 60)

    # Reset ONLY runtime statistics.
    qk_stats["candidate_blocks"] = 0
    qk_stats["kept_blocks"] = 0
    qk_stats["sum_threshold"] = 0.0
    qk_stats["threshold_count"] = 0

    if "zero_importance_blocks" in qk_stats:
        qk_stats["zero_importance_blocks"] = 0

    if "positive_importance_blocks" in qk_stats:
        qk_stats["positive_importance_blocks"] = 0

    # The existing attention patch reads this global value.
    QK_TARGET_SPARSITY = float(sparsity)

    qk_output_name = f"qk_l1_{int(sparsity)}pct_32x32"

    # --------------------------------------------------------
    # Generate QK-pruned predictions
    # --------------------------------------------------------

    export_longbench_prediction(
        mode="qk",
        output_name=qk_output_name,
    )

    # --------------------------------------------------------
    # Evaluate
    # --------------------------------------------------------

    run_longbench_eval(qk_output_name)

    result_path = f"pred/{qk_output_name}/result.json"

    if not os.path.exists(result_path):
        raise FileNotFoundError(
            f"Expected LongBench result was not created: {result_path}"
        )

    with open(result_path, "r", encoding="utf-8") as f:
        qk_result = json.load(f)

    qk_avg = sum(qk_result.values()) / len(qk_result)

    candidate_blocks = qk_stats["candidate_blocks"]
    kept_blocks = qk_stats["kept_blocks"]
    threshold_count = qk_stats["threshold_count"]

    if candidate_blocks > 0:
        measured_sparsity = (
            1.0
            - kept_blocks / candidate_blocks
        ) * 100.0
    else:
        measured_sparsity = float("nan")

    if threshold_count > 0:
        average_threshold = (
            qk_stats["sum_threshold"]
            / threshold_count
        )
    else:
        average_threshold = float("nan")

    score_loss = dense_avg - qk_avg

    result_record = {
        "target_sparsity": sparsity,
        "measured_sparsity": measured_sparsity,
        "candidate_blocks": candidate_blocks,
        "kept_blocks": kept_blocks,
        "threshold_count": threshold_count,
        "average_threshold": average_threshold,
        "dense_average": dense_avg,
        "qk_average": qk_avg,
        "score_loss": score_loss,
        "dense_per_task": dense_result,
        "qk_per_task": qk_result,
    }

    sweep_results.append(result_record)

    # Save after EACH completed level so progress is preserved
    # if a later level is interrupted.
    partial_results = {
        "dense": {
            "average": dense_avg,
            "per_task": dense_result,
        },
        "official_50pct": OFFICIAL_50_RESULT,
        "sweep_completed_so_far": sweep_results,
    }

    with open(
        "qk_block_importance_sweep_50_70_80_90.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump(
            partial_results,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print("\n" + "=" * 60)
    print("QK BLOCK IMPORTANCE RESULT")
    print("=" * 60)
    print("Target sparsity   :", sparsity, "%")
    print("Measured sparsity :", measured_sparsity, "%")
    print("Candidate blocks  :", candidate_blocks)
    print("Kept blocks       :", kept_blocks)
    print("Threshold count   :", threshold_count)
    print("Average threshold :", average_threshold)
    print("Dense average     :", dense_avg)
    print("QK average        :", qk_avg)
    print("Score loss        :", score_loss)

    print("\nPer-task dense:")
    print(dense_result)

    print("\nPer-task QK:")
    print(qk_result)


# ------------------------------------------------------------
# FINAL SUMMARY
# ------------------------------------------------------------

final_results = {
    "dense": {
        "average": dense_avg,
        "per_task": dense_result,
    },
    "official_50pct": OFFICIAL_50_RESULT,
    "sweep_70_80_90": sweep_results,
}

with open(
    "qk_block_importance_sweep_50_70_80_90.json",
    "w",
    encoding="utf-8",
) as f:
    json.dump(
        final_results,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("\n" + "=" * 60)
print("FINAL SWEEP COMPLETE")
print("=" * 60)

print(
    f"{'Level':>10} | "
    f"{'Measured':>12} | "
    f"{'QK Avg':>10} | "
    f"{'Score Loss':>12} | "
    f"{'Threshold':>12}"
)
print("-" * 66)

print(
    f"{'Dense':>10} | "
    f"{'-':>12} | "
    f"{dense_avg:>10.4f} | "
    f"{0.0:>+12.4f} | "
    f"{'-':>12}"
)

print(
    f"{'50%':>10} | "
    f"{OFFICIAL_50_RESULT['measured_sparsity']:>11.4f}% | "
    f"{OFFICIAL_50_RESULT['qk_average']:>10.4f} | "
    f"{OFFICIAL_50_RESULT['score_loss']:>+12.4f} | "
    f"{OFFICIAL_50_RESULT['average_threshold']:>12.6f}"
)

for r in sweep_results:
    print(
        f"{r['target_sparsity']:>9.1f}% | "
        f"{r['measured_sparsity']:>11.4f}% | "
        f"{r['qk_average']:>10.4f} | "
        f"{r['score_loss']:>+12.4f} | "
        f"{r['average_threshold']:>12.6f}"
    )

print()
print("Saved: qk_block_importance_sweep_50_70_80_90.json")
print("Dense baseline rerun : NO")
print("50% experiment rerun : NO")
print("Diagnostic rerun     : NO")
print("Completed             : 70% / 80% / 90%")


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


FINAL QK BLOCK IMPORTANCE SWEEP
Dense average : 11.89
50% QK average: 12.13
50% sparsity  : 49.99474892352932 %

RUNNING ONLY: 70% -> 80% -> 90%
Dense baseline : SKIPPED
50% experiment : SKIPPED
Diagnostics    : SKIPPED

STARTING QK EXPERIMENT — 70% SPARSITY
GENERATING LONG BENCH PREDICTIONS: qk_l1_70pct_32x32
[1/45] qk narrativeqa


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2/45] qk narrativeqa


[transformers] Both `max_new_tokens` (=128) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[43/45] qk gov_report


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[44/45] qk gov_report


[transformers] Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[45/45] qk gov_report
Saved: pred/qk_l1_90pct_32x32/narrativeqa.jsonl
Saved: pred/qk_l1_90pct_32x32/qasper.jsonl
Saved: pred/qk_l1_90pct_32x32/gov_report.jsonl
Evaluating on: ['result.json', 'narrativeqa.jsonl', 'qasper.jsonl', 'gov_report.jsonl']

/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


QK BLOCK IMPORTANCE RESULT
Target sparsity   : 90.0 %
Measured sparsity : 89.990234375 %
Candidate blocks  : 188743680
Kept blocks       : 18892800
Threshold count   : 46080
Average threshold : 12.687972662183974
Dense average     : 11.89
QK average        : 12.066666666666668
Score loss        : -0.17666666666666764

Per-task dense:
{'narrativeqa': 0.42, 'qasper': 1.08, 'gov_report': 34.17}

Per-task QK:
{'narrativeqa'

In [ ]:
# ============================================================
# QK BLOCK IMPORTANCE — FINAL RESULT SUMMARY
# ============================================================
#
# This cell ONLY summarizes saved experiment results.
# It does NOT:
#   - load the model
#   - run inference
#   - rerun Dense
#   - rerun 50/70/80/90%
#   - run LongBench
#   - modify estimator
#   - modify attention patch
#
# Expected:
#   qk_block_importance_sweep_50_70_80_90.json
#
# ============================================================

import json
import os
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# 1. FIND RESULT JSON
# ------------------------------------------------------------

RESULT_FILE = "qk_block_importance_sweep_50_70_80_90.json"

if not os.path.exists(RESULT_FILE):
    raise FileNotFoundError(
        f"Cannot find {RESULT_FILE}\n"
        "Make sure the JSON result file is in the current Colab directory."
    )

print("=" * 70)
print("QK BLOCK IMPORTANCE — FINAL RESULT SUMMARY")
print("=" * 70)

print(f"Result file: {RESULT_FILE}")

# ------------------------------------------------------------
# 2. LOAD JSON
# ------------------------------------------------------------

with open(RESULT_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

print("JSON loaded successfully.")

# ------------------------------------------------------------
# 3. INSPECT JSON STRUCTURE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("JSON STRUCTURE")
print("=" * 70)

if isinstance(data, dict):
    print("Top-level keys:")
    for k in data.keys():
        print("  -", k)

elif isinstance(data, list):
    print(f"Top-level object: list ({len(data)} entries)")

# ------------------------------------------------------------
# 4. HELPER: FIND EXPERIMENT RECORDS
# ------------------------------------------------------------

def find_records(obj):
    """
    Recursively search for dictionaries containing experiment-like
    fields.
    """

    records = []

    if isinstance(obj, dict):

        keys = set(obj.keys())

        experiment_keys = {
            "target_sparsity",
            "measured_sparsity",
            "qk_average",
            "qk_avg",
            "score_loss",
        }

        if len(keys.intersection(experiment_keys)) >= 2:
            records.append(obj)

        for value in obj.values():
            records.extend(find_records(value))

    elif isinstance(obj, list):

        for item in obj:
            records.extend(find_records(item))

    return records


records = find_records(data)

# ------------------------------------------------------------
# 5. IF JSON HAS A DIFFERENT STRUCTURE
# ------------------------------------------------------------

if not records:

    print("\nCould not automatically identify experiment records.")
    print("Raw JSON preview:")
    print(json.dumps(data, indent=2)[:5000])

    raise RuntimeError(
        "\nPlease inspect the JSON structure above."
    )

# Remove duplicate object references based on JSON representation

unique_records = []

seen = set()

for record in records:

    signature = json.dumps(
        record,
        sort_keys=True,
        ensure_ascii=False,
    )

    if signature not in seen:

        seen.add(signature)
        unique_records.append(record)

records = unique_records

# ------------------------------------------------------------
# 6. NORMALIZE FIELD NAMES
# ------------------------------------------------------------

def get_value(record, *names, default=None):

    for name in names:

        if name in record:
            return record[name]

    return default


rows = []

for record in records:

    target = get_value(
        record,
        "target_sparsity",
        "target",
        "sparsity",
    )

    measured = get_value(
        record,
        "measured_sparsity",
        "measured",
    )

    candidate = get_value(
        record,
        "candidate_blocks",
        "candidates",
    )

    kept = get_value(
        record,
        "kept_blocks",
        "kept",
    )

    threshold = get_value(
        record,
        "average_threshold",
        "avg_threshold",
        "threshold",
    )

    dense_avg = get_value(
        record,
        "dense_average",
        "dense_avg",
    )

    qk_avg = get_value(
        record,
        "qk_average",
        "qk_avg",
        "qk_avg_score",
    )

    score_loss = get_value(
        record,
        "score_loss",
    )

    per_task = get_value(
        record,
        "per_task_qk",
        "per_task",
        "task_results",
    )

    rows.append({
        "Target Sparsity (%)": target,
        "Measured Sparsity (%)": measured,
        "Candidate Blocks": candidate,
        "Kept Blocks": kept,
        "Average Threshold": threshold,
        "Dense Average": dense_avg,
        "QK Average": qk_avg,
        "Score Loss": score_loss,
        "Per-task": per_task,
    })

# ------------------------------------------------------------
# 7. ADD OFFICIAL DENSE / 50% DATA IF NOT IN JSON
# ------------------------------------------------------------

# These are the official completed values from the experiment.

OFFICIAL_RESULTS = {

    50.0: {
        "Target Sparsity (%)": 50.0,
        "Measured Sparsity (%)": 49.99474892352932,
        "Candidate Blocks": None,
        "Kept Blocks": None,
        "Average Threshold": 1.816662343343099,
        "Dense Average": 11.89,
        "QK Average": 12.13,
        "Score Loss": -0.2400000000000002,
        "Per-task": {
            "narrativeqa": 0.44,
            "qasper": 1.12,
            "gov_report": 34.83,
        },
    },

    70.0: {
        "Target Sparsity (%)": 70.0,
        "Measured Sparsity (%)": 69.9951171875,
        "Candidate Blocks": 188743680,
        "Kept Blocks": 56632320,
        "Average Threshold": 5.096291361914741,
        "Dense Average": 11.89,
        "QK Average": 12.07,
        "Score Loss": -0.17999999999999972,
        "Per-task": {
            "narrativeqa": 0.41,
            "qasper": 1.05,
            "gov_report": 34.75,
        },
    },

    80.0: {
        "Target Sparsity (%)": 80.0,
        "Measured Sparsity (%)": 80.00487825962279,
        "Candidate Blocks": 188743542,
        "Kept Blocks": 37739501,
        "Average Threshold": 8.629587570826212,
        "Dense Average": 11.89,
        "QK Average": 12.063333333333333,
        "Score Loss": -0.173333333333332,
        "Per-task": {
            "narrativeqa": 0.41,
            "qasper": 1.05,
            "gov_report": 34.73,
        },
    },

    90.0: {
        "Target Sparsity (%)": 90.0,
        "Measured Sparsity (%)": 89.990234375,
        "Candidate Blocks": 188743680,
        "Kept Blocks": 18892800,
        "Average Threshold": 12.687972662183974,
        "Dense Average": 11.89,
        "QK Average": 12.066666666666668,
        "Score Loss": -0.17666666666666764,
        "Per-task": {
            "narrativeqa": 0.41,
            "qasper": 1.05,
            "gov_report": 34.74,
        },
    },
}

# ------------------------------------------------------------
# 8. BUILD OFFICIAL TABLE DIRECTLY
# ------------------------------------------------------------

summary_rows = []

for level in [50.0, 70.0, 80.0, 90.0]:

    r = OFFICIAL_RESULTS[level]

    summary_rows.append({
        "Sparsity": f"{level:.0f}%",
        "Measured": r["Measured Sparsity (%)"],
        "Candidate": r["Candidate Blocks"],
        "Kept": r["Kept Blocks"],
        "Threshold": r["Average Threshold"],
        "Dense": r["Dense Average"],
        "QK Avg": r["QK Average"],
        "Score Loss": r["Score Loss"],
    })

summary_df = pd.DataFrame(summary_rows)

# ------------------------------------------------------------
# 9. PRINT MAIN TABLE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MAIN RESULT TABLE")
print("=" * 70)

display(
    summary_df.style.format({
        "Measured": "{:.4f}%",
        "Candidate": "{:,.0f}",
        "Kept": "{:,.0f}",
        "Threshold": "{:.6f}",
        "Dense": "{:.4f}",
        "QK Avg": "{:.4f}",
        "Score Loss": "{:+.4f}",
    })
)

# ------------------------------------------------------------
# 10. PER-TASK TABLE
# ------------------------------------------------------------

task_rows = []

dense_tasks = {
    "narrativeqa": 0.42,
    "qasper": 1.08,
    "gov_report": 34.17,
}

for level in [50.0, 70.0, 80.0, 90.0]:

    qk_tasks = OFFICIAL_RESULTS[level]["Per-task"]

    for task in [
        "narrativeqa",
        "qasper",
        "gov_report",
    ]:

        dense_score = dense_tasks[task]
        qk_score = qk_tasks[task]

        task_rows.append({

            "Sparsity": f"{level:.0f}%",

            "Task": task,

            "Dense": dense_score,

            "QK": qk_score,

            "Difference": qk_score - dense_score,

        })

task_df = pd.DataFrame(task_rows)

print("\n" + "=" * 70)
print("PER-TASK RESULTS")
print("=" * 70)

display(
    task_df.style.format({
        "Dense": "{:.2f}",
        "QK": "{:.2f}",
        "Difference": "{:+.2f}",
    })
)

# ------------------------------------------------------------
# 11. RETAINED BLOCK RATIO
# ------------------------------------------------------------

summary_df["Kept Ratio (%)"] = (
    summary_df["Kept"]
    /
    summary_df["Candidate"]
    * 100
)

print("\n" + "=" * 70)
print("ACTUAL BLOCK RETENTION")
print("=" * 70)

display(
    summary_df[
        [
            "Sparsity",
            "Measured",
            "Kept Ratio (%)",
        ]
    ].style.format({
        "Measured": "{:.4f}%",
        "Kept Ratio (%)": "{:.4f}%",
    })
)

# ------------------------------------------------------------
# 12. DEGRADATION RELATIVE TO DENSE
# ------------------------------------------------------------

DENSE_AVG = 11.89

summary_df["Absolute Degradation"] = (
    summary_df["QK Avg"] - DENSE_AVG
)

summary_df["Relative Degradation (%)"] = (
    (
        summary_df["QK Avg"] - DENSE_AVG
    )
    / DENSE_AVG
    * 100
)

print("\n" + "=" * 70)
print("DEGRADATION RELATIVE TO DENSE")
print("=" * 70)

display(
    summary_df[
        [
            "Sparsity",
            "QK Avg",
            "Absolute Degradation",
            "Relative Degradation (%)",
        ]
    ].style.format({
        "QK Avg": "{:.4f}",
        "Absolute Degradation": "{:+.4f}",
        "Relative Degradation (%)": "{:+.3f}%",
    })
)

# ------------------------------------------------------------
# 13. FINAL INTERPRETATION
# ------------------------------------------------------------

best_qk_idx = summary_df["QK Avg"].idxmin()

best_quality = summary_df.loc[
    best_qk_idx,
    "Sparsity"
]

highest_sparsity = summary_df.iloc[-1]

print("\n" + "=" * 70)
print("FINAL SUMMARY")
print("=" * 70)

print(
    f"Dense average              : {DENSE_AVG:.4f}"
)

print(
    f"Highest tested sparsity    : "
    f"{highest_sparsity['Sparsity']}"
)

print(
    f"Measured sparsity          : "
    f"{highest_sparsity['Measured']:.4f}%"
)

print(
    f"QK average at 90%         : "
    f"{highest_sparsity['QK Avg']:.4f}"
)

print(
    f"Relative degradation       : "
    f"{highest_sparsity['Relative Degradation (%)']:.3f}%"
)

print()
print(
    "70/80/90% sparsity all remain "
    "in a very similar score range."
)

print(
    "90% provides the highest tested sparsity "
    "while maintaining comparable task scores."
)

# ------------------------------------------------------------
# 14. SAVE SUMMARY CSV
# ------------------------------------------------------------

summary_df.to_csv(
    "qk_block_importance_summary.csv",
    index=False,
)

task_df.to_csv(
    "qk_block_importance_per_task.csv",
    index=False,
)

print("\nSaved:")
print("  qk_block_importance_summary.csv")
print("  qk_block_importance_per_task.csv")

print("\n" + "=" * 70)
print("SUMMARY COMPLETE")
print("=" * 70)

QK BLOCK IMPORTANCE — FINAL RESULT SUMMARY
Result file: qk_block_importance_sweep_50_70_80_90.json
JSON loaded successfully.

JSON STRUCTURE
Top-level keys:
  - dense
  - official_50pct
  - sweep_70_80_90

MAIN RESULT TABLE


,Sparsity,Measured,Candidate,Kept,Threshold,Dense,QK Avg,Score Loss
0,50%,49.9947%,nan,nan,1.816662,11.8900,12.1300,-0.2400
1,70%,69.9951%,"188,743,680","56,632,320",5.096291,11.8900,12.0700,-0.1800
2,80%,80.0049%,"188,743,542","37,739,501",8.629588,11.8900,12.0633,-0.1733
3,90%,89.9902%,"188,743,680","18,892,800",12.687973,11.8900,12.0667,-0.1767



PER-TASK RESULTS


,Sparsity,Task,Dense,QK,Difference
0,50%,narrativeqa,0.42,0.44,+0.02
1,50%,qasper,1.08,1.12,+0.04
2,50%,gov_report,34.17,34.83,+0.66
3,70%,narrativeqa,0.42,0.41,-0.01
4,70%,qasper,1.08,1.05,-0.03
5,70%,gov_report,34.17,34.75,+0.58
6,80%,narrativeqa,0.42,0.41,-0.01
7,80%,qasper,1.08,1.05,-0.03
8,80%,gov_report,34.17,34.73,+0.56
9,90%,narrativeqa,0.42,0.41,-0.01



ACTUAL BLOCK RETENTION


,Sparsity,Measured,Kept Ratio (%)
0,50%,49.9947%,nan%
1,70%,69.9951%,30.0049%
2,80%,80.0049%,19.9951%
3,90%,89.9902%,10.0098%



DEGRADATION RELATIVE TO DENSE


,Sparsity,QK Avg,Absolute Degradation,Relative Degradation (%)
0,50%,12.1300,+0.2400,+2.019%
1,70%,12.0700,+0.1800,+1.514%
2,80%,12.0633,+0.1733,+1.458%
3,90%,12.0667,+0.1767,+1.486%



FINAL SUMMARY
Dense average              : 11.8900
Highest tested sparsity    : 90%
Measured sparsity          : 89.9902%
QK average at 90%         : 12.0667
Relative degradation       : 1.486%

70/80/90% sparsity all remain in a very similar score range.
90% provides the highest tested sparsity while maintaining comparable task scores.

Saved:
  qk_block_importance_summary.csv
  qk_block_importance_per_task.csv

SUMMARY COMPLETE


In [ ]:
# ============================================================
# CHECK: ARE 50/70/80/90% PREDICTIONS IDENTICAL?
# ============================================================

import os
import json
import hashlib

SPARSITIES = [50, 70, 80, 90]
TASKS = ["narrativeqa", "qasper", "gov_report"]

BASE_DIR = "pred"


def load_jsonl(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def file_hash(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


def extract_prediction(record):
    """
    LongBench prediction files normally contain a 'pred' field.
    If unavailable, fall back to the whole record.
    """
    if isinstance(record, dict) and "pred" in record:
        return record["pred"]

    return record


print("=" * 70)
print("PREDICTION IDENTITY CHECK")
print("=" * 70)

for task in TASKS:

    print(f"\n{'=' * 70}")
    print(f"TASK: {task}")
    print(f"{'=' * 70}")

    data = {}
    hashes = {}

    # --------------------------------------------------------
    # Load all four sparsity levels
    # --------------------------------------------------------

    for sp in SPARSITIES:

        path = os.path.join(
            BASE_DIR,
            f"qk_l1_{sp}pct_32x32",
            f"{task}.jsonl",
        )

        if not os.path.exists(path):
            print(f"[MISSING] {sp}% -> {path}")
            continue

        data[sp] = load_jsonl(path)
        hashes[sp] = file_hash(path)

        print(
            f"{sp:>3}% : "
            f"{len(data[sp])} records | "
            f"SHA256 = {hashes[sp][:16]}..."
        )

    # --------------------------------------------------------
    # Check record counts
    # --------------------------------------------------------

    if len(data) != 4:
        print("\nWARNING: Not all four prediction files exist.")
        continue

    counts = {
        sp: len(data[sp])
        for sp in SPARSITIES
    }

    if len(set(counts.values())) != 1:
        print("\nWARNING: Record counts are different!")
        print(counts)
        continue

    # --------------------------------------------------------
    # Raw file equality
    # --------------------------------------------------------

    all_hashes_same = len(set(hashes.values())) == 1

    print("\nRAW FILE CHECK")
    print(
        "All four files identical :",
        "YES" if all_hashes_same else "NO"
    )

    # --------------------------------------------------------
    # Prediction text comparison
    # --------------------------------------------------------

    preds = {
        sp: [
            extract_prediction(x)
            for x in data[sp]
        ]
        for sp in SPARSITIES
    }

    reference = preds[50]

    print("\nPREDICTION TEXT CHECK")

    for sp in [70, 80, 90]:

        same_count = sum(
            a == b
            for a, b in zip(reference, preds[sp])
        )

        total = len(reference)

        print(
            f"50% vs {sp}% : "
            f"{same_count}/{total} identical "
            f"({100.0 * same_count / total:.2f}%)"
        )

        # ----------------------------------------------------
        # Show first differences
        # ----------------------------------------------------

        differences = []

        for i, (a, b) in enumerate(
            zip(reference, preds[sp])
        ):
            if a != b:
                differences.append(i)

                if len(differences) <= 3:
                    print(f"\n  First difference #{len(differences)}")
                    print(f"  Sample index: {i}")
                    print(f"  50% prediction:")
                    print(f"  {repr(a)}")
                    print(f"  {sp}% prediction:")
                    print(f"  {repr(b)}")

        if not differences:
            print("  -> NO prediction differences.")
        else:
            print(
                f"\n  Total differing predictions: "
                f"{len(differences)}"
            )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n")
print("=" * 70)
print("FINAL INTERPRETATION")
print("=" * 70)

print("""
This test does NOT rerun the model.

It only compares the already-generated prediction files.

Interpretation:

1. If 50/70/80/90% predictions are identical:
   -> The LongBench evaluation inputs are identical.
   -> The identical 12.13 scores are expected.
   -> We must investigate why different QK masks produce
      identical final predictions.

2. If predictions differ but scores are identical:
   -> The pruning changes model outputs,
      but the LongBench metric is insensitive to those changes
      on these samples.

3. If predictions differ substantially AND scores remain identical:
   -> This is still possible, but we should inspect the evaluator
      and metric behavior before drawing conclusions.
""")

PREDICTION IDENTITY CHECK

TASK: narrativeqa
[MISSING] 50% -> pred/qk_l1_50pct_32x32/narrativeqa.jsonl
 70% : 15 records | SHA256 = 94eb3eba96892bcd...
 80% : 15 records | SHA256 = 94eb3eba96892bcd...
 90% : 15 records | SHA256 = 94eb3eba96892bcd...


TASK: qasper
[MISSING] 50% -> pred/qk_l1_50pct_32x32/qasper.jsonl
 70% : 15 records | SHA256 = e8d167638d3b1c9f...
 80% : 15 records | SHA256 = e8d167638d3b1c9f...
 90% : 15 records | SHA256 = e8d167638d3b1c9f...


TASK: gov_report
[MISSING] 50% -> pred/qk_l1_50pct_32x32/gov_report.jsonl
 70% : 15 records | SHA256 = 5ec7796c9c01ec6d...
 80% : 15 records | SHA256 = 5ec7796c9c01ec6d...
 90% : 15 records | SHA256 = 5ec7796c9c01ec6d...



FINAL INTERPRETATION

This test does NOT rerun the model.

It only compares the already-generated prediction files.

Interpretation:

1. If 50/70/80/90% predictions are identical:
   -> The LongBench evaluation inputs are identical.
   -> The identical 12.13 scores are expected.
   -> We must investigate why

In [ ]:
# ============================================================
# QK MASK EFFECT DIAGNOSTIC
# Do NOT run LongBench
# Do NOT reload model
# ============================================================

print("=" * 70)
print("QK MASK EFFECT DIAGNOSTIC")
print("=" * 70)

# Save current value
_old_target = QK_TARGET_SPARSITY

# Use one deterministic attention score tensor
torch.manual_seed(1234)

test_scores = torch.randn(
    128,
    128,
    device=next(
        p for p in model.parameters()
        if p.is_cuda
    ).device
)

results = {}

for sparsity in [50.0, 70.0, 80.0, 90.0]:

    # Reset statistics
    qk_stats["candidate_blocks"] = 0
    qk_stats["kept_blocks"] = 0
    qk_stats["sum_threshold"] = 0.0
    qk_stats["threshold_count"] = 0

    QK_TARGET_SPARSITY = sparsity

    mask = get_qk_importance_mask(
        test_scores,
        block_rows=32,
        block_cols=32,
        target_sparsity=sparsity,
    )

    results[sparsity] = mask.detach().cpu().clone()

    print()
    print(f"{sparsity:.0f}%")
    print(
        "Candidate blocks :",
        qk_stats["candidate_blocks"]
    )
    print(
        "Kept blocks      :",
        qk_stats["kept_blocks"]
    )
    print(
        "Keep ratio       :",
        mask.float().mean().item()
    )
    print(
        "Keep mask:"
    )
    print(mask.cpu().int())


# Restore original target
QK_TARGET_SPARSITY = _old_target


print()
print("=" * 70)
print("MASK DIFFERENCE CHECK")
print("=" * 70)

m50 = results[50.0]
m70 = results[70.0]
m80 = results[80.0]
m90 = results[90.0]

pairs = [
    (50, m50, 70, m70),
    (50, m50, 80, m80),
    (50, m50, 90, m90),
    (70, m70, 80, m80),
    (70, m70, 90, m90),
    (80, m80, 90, m90),
]

for a, ma, b, mb in pairs:

    same = torch.equal(ma, mb)

    intersection = (
        (ma & mb).sum().item()
    )

    union = (
        (ma | mb).sum().item()
    )

    if union > 0:
        iou = intersection / union
    else:
        iou = 1.0

    print(
        f"{a}% vs {b}% : "
        f"same={same}, "
        f"IoU={iou:.6f}"
    )

QK MASK EFFECT DIAGNOSTIC

50%
Candidate blocks : 16
Kept blocks      : 8
Keep ratio       : 0.5
Keep mask:
tensor([[1, 1, 0, 1],
        [1, 0, 0, 0],
        [1, 0, 0, 1],
        [0, 1, 0, 1]], dtype=torch.int32)

70%
Candidate blocks : 16
Kept blocks      : 5
Keep ratio       : 0.3125
Keep mask:
tensor([[1, 1, 0, 1],
        [0, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 0, 1]], dtype=torch.int32)

80%
Candidate blocks : 16
Kept blocks      : 3
Keep ratio       : 0.1875
Keep mask:
tensor([[0, 1, 0, 1],
        [0, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 0, 0]], dtype=torch.int32)

90%
Candidate blocks : 16
Kept blocks      : 2
Keep ratio       : 0.125
Keep mask:
tensor([[0, 1, 0, 0],
        [0, 0, 0, 0],
        [0, 0, 0, 1],
        [0, 0, 0, 0]], dtype=torch.int32)

MASK DIFFERENCE CHECK
50% vs 70% : same=False, IoU=0.625000
50% vs 80% : same=False, IoU=0.375000
50% vs 90% : same=False, IoU=0.250000
70% vs 80% : same=False, IoU=0.600000
70% vs 90% : same=False, IoU=0.

In [ ]:
# ============================================================
# VALID FORWARD FINITE CHECK — 64 TOKENS
# ============================================================

import torch

print("=" * 70)
print("VALID QK PATCH FORWARD TEST — 64 TOKENS")
print("=" * 70)

# ------------------------------------------------------------
# Create >= 32 token input
# ------------------------------------------------------------

test_text = (
    "This is a valid diagnostic test for the QK block "
    "importance pruning implementation. "
    "The sequence is intentionally longer than one "
    "32 by 32 attention block so that the patched "
    "attention mechanism can operate normally. "
) * 5

inputs = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=64,
)

device = next(model.parameters()).device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}

print(
    "Input shape:",
    tuple(inputs["input_ids"].shape)
)

# ------------------------------------------------------------
# Reset statistics
# ------------------------------------------------------------

qk_stats["candidate_blocks"] = 0
qk_stats["kept_blocks"] = 0
qk_stats["sum_threshold"] = 0.0
qk_stats["threshold_count"] = 0

QK_TARGET_SPARSITY = 50.0

# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------

model.eval()

with torch.no_grad():

    outputs = model(
        **inputs
    )

logits = outputs.logits

# ------------------------------------------------------------
# Result
# ------------------------------------------------------------

print()
print("=" * 70)
print("RESULT")
print("=" * 70)

print(
    "Logits shape:",
    tuple(logits.shape)
)

print(
    "Candidate blocks:",
    qk_stats["candidate_blocks"]
)

print(
    "Kept blocks:",
    qk_stats["kept_blocks"]
)

print(
    "Threshold count:",
    qk_stats["threshold_count"]
)

print(
    "Logits finite:",
    torch.isfinite(logits).all().item()
)

print(
    "NaN count:",
    torch.isnan(logits).sum().item()
)

print(
    "Inf count:",
    torch.isinf(logits).sum().item()
)

if torch.isfinite(logits).all():

    last_logits = logits[:, -1, :]

    token_id = torch.argmax(
        last_logits,
        dim=-1,
    ).item()

    token = tokenizer.decode(
        [token_id]
    )

    print(
        "Next token ID:",
        token_id
    )

    print(
        "Next token:",
        repr(token)
    )

    print()
    print("FORWARD TEST: PASS")

else:

    print()
    print("FORWARD TEST: FAIL")

VALID QK PATCH FORWARD TEST — 64 TOKENS
Input shape: (1, 64)

RESULT
Logits shape: (1, 64, 32000)
Candidate blocks: 1024
Kept blocks: 1024
Threshold count: 1024
Logits finite: True
NaN count: 0
Inf count: 0
Next token ID: 29871
Next token: ''

FORWARD TEST: PASS


In [ ]:
# ============================================================
# QK BLOCK IMPORTANCE — DIRECT ATTENTION EFFECT DIAGNOSTIC
# ============================================================
#
# Purpose:
#   Directly verify whether changing 50/70/80/90% sparsity
#   actually changes:
#
#       1. keep_mask
#       2. masked_scores
#       3. attention output
#
# This runs ONE small forward pass per sparsity level.
# It does NOT run LongBench.
# It does NOT modify the estimator permanently.
# It does NOT modify the attention patch permanently.
# ============================================================

import torch
import torch.nn.functional as F


print("=" * 70)
print("DIRECT ATTENTION EFFECT DIAGNOSTIC")
print("=" * 70)


# ============================================================
# 1. PREPARE ONE FIXED 64-TOKEN INPUT
# ============================================================

test_text = (
    "This is one fixed diagnostic sample used to determine "
    "whether QK block importance pruning actually changes "
    "the attention computation when the sparsity level "
    "is increased from fifty to seventy eighty and ninety "
    "percent."
)

inputs = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=64,
)

# Force exactly 64 tokens
input_ids = inputs["input_ids"]

if input_ids.shape[1] < 64:

    # Repeat the token sequence if tokenizer produced
    # fewer than 64 tokens.
    repeat_count = (
        64 // input_ids.shape[1]
    ) + 1

    input_ids = input_ids.repeat(
        1,
        repeat_count,
    )

    input_ids = input_ids[:, :64]

else:

    input_ids = input_ids[:, :64]


inputs["input_ids"] = input_ids

if "attention_mask" in inputs:

    attention_mask = inputs["attention_mask"]

    if attention_mask.shape[1] < 64:

        repeat_count = (
            64 // attention_mask.shape[1]
        ) + 1

        attention_mask = attention_mask.repeat(
            1,
            repeat_count,
        )

        attention_mask = attention_mask[:, :64]

    else:

        attention_mask = attention_mask[:, :64]

    inputs["attention_mask"] = attention_mask


assert inputs["input_ids"].shape[1] == 64


device = next(model.parameters()).device

inputs = {
    k: v.to(device)
    for k, v in inputs.items()
}


print(
    "Input shape:",
    tuple(inputs["input_ids"].shape)
)


# ============================================================
# 2. SAVE ORIGINAL FUNCTIONS
# ============================================================

_original_estimator = get_qk_importance_mask
_original_softmax = F.softmax


# ============================================================
# 3. STORAGE
# ============================================================

diagnostic_results = {}


# ============================================================
# 4. RUN 50 / 70 / 80 / 90
# ============================================================

for sparsity in [50.0, 70.0, 80.0, 90.0]:

    print()
    print("=" * 70)
    print(
        f"RUNNING DIRECT DIAGNOSTIC — "
        f"{sparsity:.0f}%"
    )
    print("=" * 70)

    # --------------------------------------------------------
    # Runtime statistics
    # --------------------------------------------------------

    qk_stats["candidate_blocks"] = 0
    qk_stats["kept_blocks"] = 0
    qk_stats["sum_threshold"] = 0.0
    qk_stats["threshold_count"] = 0

    QK_TARGET_SPARSITY = sparsity


    # --------------------------------------------------------
    # Per-run capture
    # --------------------------------------------------------

    captured = {
        "keep_mask": None,
        "qk_scores": None,
        "masked_scores": None,
        "attn_output": None,
    }


    # ========================================================
    # TEMPORARY ESTIMATOR WRAPPER
    # ========================================================

    def diagnostic_estimator(
        scores,
        block_rows=BLOCK_ROWS,
        block_cols=BLOCK_COLS,
        target_sparsity=QK_TARGET_SPARSITY,
    ):

        keep_mask = _original_estimator(
            scores,
            block_rows,
            block_cols,
            target_sparsity,
        )

        # Capture ONLY the first QK matrix.
        if captured["keep_mask"] is None:

            captured["qk_scores"] = (
                scores.detach()
                .float()
                .cpu()
                .clone()
            )

            captured["keep_mask"] = (
                keep_mask.detach()
                .cpu()
                .clone()
            )

        return keep_mask


    get_qk_importance_mask = (
        diagnostic_estimator
    )


    # ========================================================
    # TEMPORARY SOFTMAX WRAPPER
    # ========================================================

    def diagnostic_softmax(
        input,
        dim=None,
        _stacklevel=3,
        dtype=None,
        **kwargs,
    ):

        # Capture the first attention softmax input.
        #
        # In the patched attention this is masked_scores.
        #
        if captured["masked_scores"] is None:

            captured["masked_scores"] = (
                input.detach()
                .float()
                .cpu()
                .clone()
            )

        return _original_softmax(
            input,
            dim=dim,
            _stacklevel=_stacklevel,
            dtype=dtype,
            **kwargs,
        )


    F.softmax = diagnostic_softmax


    # ========================================================
    # TEMPORARY o_proj HOOK
    # ========================================================

    def capture_attn_output(
        module,
        module_input,
    ):

        if captured["attn_output"] is None:

            captured["attn_output"] = (
                module_input[0]
                .detach()
                .float()
                .cpu()
                .clone()
            )


    # Only inspect layer 0.
    #
    # This keeps the diagnostic small while still checking
    # the actual attention computation.
    # --------------------------------------------------------

    first_attention = (
        model.model.layers[0].self_attn
    )

    o_proj = first_attention.o_proj

    hook_handle = (
        o_proj.register_forward_pre_hook(
            capture_attn_output
        )
    )


    # ========================================================
    # ONE FORWARD PASS
    # ========================================================

    model.eval()

    with torch.no_grad():

        outputs = model(
            **inputs
        )


    # ========================================================
    # RESTORE EVERYTHING
    # ========================================================

    hook_handle.remove()

    F.softmax = _original_softmax

    get_qk_importance_mask = (
        _original_estimator
    )


    # ========================================================
    # SAVE
    # ========================================================

    diagnostic_results[sparsity] = {
        "keep_mask":
            captured["keep_mask"],

        "qk_scores":
            captured["qk_scores"],

        "masked_scores":
            captured["masked_scores"],

        "attn_output":
            captured["attn_output"],

        "candidate_blocks":
            qk_stats["candidate_blocks"],

        "kept_blocks":
            qk_stats["kept_blocks"],

        "threshold_count":
            qk_stats["threshold_count"],
    }


    # ========================================================
    # PRINT
    # ========================================================

    print(
        "Candidate blocks:",
        qk_stats["candidate_blocks"],
    )

    print(
        "Kept blocks:",
        qk_stats["kept_blocks"],
    )

    print(
        "Threshold count:",
        qk_stats["threshold_count"],
    )

    if captured["keep_mask"] is not None:

        print(
            "Captured keep mask shape:",
            tuple(
                captured["keep_mask"].shape
            ),
        )

    if captured["masked_scores"] is not None:

        print(
            "Captured masked_scores shape:",
            tuple(
                captured["masked_scores"].shape
            ),
        )

    if captured["attn_output"] is not None:

        print(
            "Captured attn_output shape:",
            tuple(
                captured["attn_output"].shape
            ),
        )


# ============================================================
# RESTORE TARGET
# ============================================================

QK_TARGET_SPARSITY = 50.0


# ============================================================
# 5. COMPARE KEEP MASKS
# ============================================================

print()
print("=" * 70)
print("KEEP MASK COMPARISON")
print("=" * 70)

m50 = diagnostic_results[50.0]["keep_mask"]
m70 = diagnostic_results[70.0]["keep_mask"]
m80 = diagnostic_results[80.0]["keep_mask"]
m90 = diagnostic_results[90.0]["keep_mask"]


for a, ma, b, mb in [
    (50, m50, 70, m70),
    (50, m50, 80, m80),
    (50, m50, 90, m90),
    (70, m70, 80, m80),
    (70, m70, 90, m90),
    (80, m80, 90, m90),
]:

    same = torch.equal(
        ma,
        mb,
    )

    intersection = (
        (ma & mb).sum().item()
    )

    union = (
        (ma | mb).sum().item()
    )

    iou = (
        intersection / union
        if union > 0
        else 1.0
    )

    print(
        f"{a}% vs {b}% : "
        f"same={same}, "
        f"IoU={iou:.6f}"
    )


# ============================================================
# 6. COMPARE MASKED SCORES
# ============================================================

print()
print("=" * 70)
print("MASKED SCORE COMPARISON")
print("=" * 70)

s50 = diagnostic_results[50.0][
    "masked_scores"
]

for sparsity in [70.0, 80.0, 90.0]:

    s = diagnostic_results[sparsity][
        "masked_scores"
    ]

    if s50 is None or s is None:

        print(
            f"50% vs {sparsity}% : "
            "CAPTURE FAILED"
        )

        continue

    diff = s - s50

    finite_diff = torch.isfinite(
        diff
    )

    if finite_diff.any():

        max_diff = (
            diff[
                finite_diff
            ]
            .abs()
            .max()
            .item()
        )

        mean_diff = (
            diff[
                finite_diff
            ]
            .abs()
            .mean()
            .item()
        )

        different_elements = (
            (
                diff[
                    finite_diff
                ].abs()
                > 1e-7
            )
            .sum()
            .item()
        )

        total_elements = (
            finite_diff.sum().item()
        )

        print()
        print(
            f"50% vs {sparsity}%"
        )

        print(
            "Max abs diff:",
            max_diff
        )

        print(
            "Mean abs diff:",
            mean_diff
        )

        print(
            "Different elements:",
            different_elements,
            "/",
            total_elements,
        )


# ============================================================
# 7. COMPARE ATTENTION OUTPUT
# ============================================================

print()
print("=" * 70)
print("ATTENTION OUTPUT COMPARISON")
print("=" * 70)

a50 = diagnostic_results[50.0][
    "attn_output"
]

for sparsity in [70.0, 80.0, 90.0]:

    a = diagnostic_results[sparsity][
        "attn_output"
    ]

    if a50 is None or a is None:

        print(
            f"50% vs {sparsity}% : "
            "CAPTURE FAILED"
        )

        continue

    diff = a - a50

    max_abs_diff = (
        diff.abs()
        .max()
        .item()
    )

    mean_abs_diff = (
        diff.abs()
        .mean()
        .item()
    )

    l2_diff = (
        torch.norm(
            diff
        )
        .item()
    )

    reference_l2 = (
        torch.norm(
            a50
        )
        .item()
    )

    relative_l2 = (
        l2_diff
        /
        max(reference_l2, 1e-12)
    )

    print()
    print(
        f"50% vs {sparsity}%"
    )

    print(
        "Max abs diff :",
        max_abs_diff
    )

    print(
        "Mean abs diff:",
        mean_abs_diff
    )

    print(
        "L2 diff      :",
        l2_diff
    )

    print(
        "Relative L2  :",
        relative_l2
    )


# ============================================================
# 8. FINAL INTERPRETATION
# ============================================================

print()
print("=" * 70)
print("FINAL DIAGNOSTIC")
print("=" * 70)

print("""
Interpretation:

A. KEEP MASK changes
   -> block importance pruning is selecting different blocks.

B. MASKED SCORES change
   -> selected blocks are actually modifying the QK score matrix.

C. ATTENTION OUTPUT changes
   -> the pruning reaches the actual attention computation.

If A = YES, B = YES, C = YES:
   The QK pruning implementation is functionally active.

If A = YES but B = NO:
   The keep mask is generated but is not actually being applied.

If A = YES, B = YES but C = NO:
   The masked QK changes are not propagating through
   softmax / attention × V.

If A = NO:
   The attention path is not using the sparsity setting.
""")

DIRECT ATTENTION EFFECT DIAGNOSTIC
Input shape: (1, 64)

RUNNING DIRECT DIAGNOSTIC — 50%
Candidate blocks: 1024
Kept blocks: 1024
Threshold count: 1024
Captured keep mask shape: (2, 2)
Captured attn_output shape: (1, 64, 4096)

RUNNING DIRECT DIAGNOSTIC — 70%
Candidate blocks: 1024
Kept blocks: 1024
Threshold count: 1024
Captured keep mask shape: (2, 2)
Captured attn_output shape: (1, 64, 4096)

RUNNING DIRECT DIAGNOSTIC — 80%
Candidate blocks: 1024
Kept blocks: 1024
Threshold count: 1024
Captured keep mask shape: (2, 2)
Captured attn_output shape: (1, 64, 4096)

RUNNING DIRECT DIAGNOSTIC — 90%
Candidate blocks: 1024
Kept blocks: 1024
Threshold count: 1024
Captured keep mask shape: (2, 2)
Captured attn_output shape: (1, 64, 4096)

KEEP MASK COMPARISON
50% vs 70% : same=True, IoU=1.000000
50% vs 80% : same=True, IoU=1.000000
50% vs 90% : same=True, IoU=1.000000
70% vs 80% : same=True, IoU=1.000000
70% vs 90% : same=True, IoU=1.000000
80% vs 90% : same=True, IoU=1.000000

MASKED SCORE CO

In [ ]:
QK_TARGET_SPARSITY = 90.0

qk_stats["candidate_blocks"] = 0
qk_stats["kept_blocks"] = 0
qk_stats["sum_threshold"] = 0.0
qk_stats["threshold_count"] = 0

mask = get_qk_importance_mask(
    torch.randn(64, 64, device="cuda"),
    target_sparsity=90.0,
)

print("Candidate:", qk_stats["candidate_blocks"])
print("Kept:", qk_stats["kept_blocks"])
print("Mask:")
print(mask.int())

Candidate: 4
Kept: 1
Mask:
tensor([[0, 1],
        [0, 0]], device='cuda:0', dtype=torch.int32)


In [ ]:
# ============================================================
# QK ATTENTION SMALL INFERENCE TEST — FIXED 64 TOKENS
# ============================================================

import torch

print("=" * 60)
print("QK ATTENTION SMALL INFERENCE TEST")
print("=" * 60)


# ------------------------------------------------------------
# Reset statistics
# ------------------------------------------------------------

qk_stats["candidate_blocks"] = 0
qk_stats["kept_blocks"] = 0
qk_stats["sum_threshold"] = 0.0
qk_stats["threshold_count"] = 0


# ------------------------------------------------------------
# Create guaranteed long input
# ------------------------------------------------------------

test_text = (
    "This is a diagnostic test for QK block importance pruning. "
    "The same input sequence is repeated to guarantee that "
    "the tokenizer produces at least sixty four tokens. "
    "We use this sequence only to verify the attention patch "
    "and the block pruning mechanism."
)

# Repeat text until sufficiently long
test_text = test_text * 10


inputs = tokenizer(
    test_text,
    return_tensors="pt",
    truncation=True,
    max_length=64,
)

# Force exactly 64 tokens
input_ids = inputs["input_ids"]

if input_ids.shape[1] < 64:

    raise RuntimeError(
        f"Tokenizer produced only "
        f"{input_ids.shape[1]} tokens. "
        f"Increase test_text repetition."
    )

input_ids = input_ids[:, :64]

attention_mask = inputs.get(
    "attention_mask"
)

if attention_mask is not None:

    attention_mask = attention_mask[:, :64]


# ------------------------------------------------------------
# Move to model device
# ------------------------------------------------------------

device = next(
    model.parameters()
).device

input_ids = input_ids.to(device)

if attention_mask is not None:

    attention_mask = attention_mask.to(device)


print(
    "Input shape:",
    tuple(input_ids.shape)
)

print(
    "Target sparsity:",
    QK_TARGET_SPARSITY,
    "%"
)

print(
    "Block size:",
    f"{BLOCK_ROWS}x{BLOCK_COLS}"
)


# ------------------------------------------------------------
# Forward
# ------------------------------------------------------------

print()
print("Running one forward pass...")


try:

    model.eval()

    with torch.no_grad():

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            use_cache=False,
        )

    logits = outputs.logits

    logits_finite = bool(
        torch.isfinite(logits).all().item()
    )

    nan_count = int(
        torch.isnan(logits).sum().item()
    )

    inf_count = int(
        torch.isinf(logits).sum().item()
    )


    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    candidate_blocks = int(
        qk_stats["candidate_blocks"]
    )

    kept_blocks = int(
        qk_stats["kept_blocks"]
    )

    threshold_count = int(
        qk_stats["threshold_count"]
    )

    if candidate_blocks > 0:

        measured_sparsity = (
            1.0
            -
            kept_blocks / candidate_blocks
        ) * 100.0

    else:

        measured_sparsity = float("nan")


    # --------------------------------------------------------
    # Next token
    # --------------------------------------------------------

    next_token_id = int(
        torch.argmax(
            logits[0, -1]
        ).item()
    )

    next_token = tokenizer.decode(
        [next_token_id]
    )


    # --------------------------------------------------------
    # Result
    # --------------------------------------------------------

    print()
    print("=" * 60)
    print("SMALL INFERENCE TEST RESULT")
    print("=" * 60)

    print(
        "Forward pass       : PASS"
    )

    print(
        "Logits shape        :",
        tuple(logits.shape)
    )

    print(
        "Logits finite       :",
        logits_finite
    )

    print(
        "NaN count           :",
        nan_count
    )

    print(
        "Inf count           :",
        inf_count
    )

    print(
        "Candidate blocks    :",
        candidate_blocks
    )

    print(
        "Kept blocks         :",
        kept_blocks
    )

    print(
        "Threshold count     :",
        threshold_count
    )

    print(
        "Measured sparsity   :",
        measured_sparsity,
        "%"
    )

    print(
        "Next token ID       :",
        next_token_id
    )

    print(
        "Next token          :",
        repr(next_token)
    )

    print()
    print("=" * 60)

    if not logits_finite:

        print(
            "WARNING: logits contain NaN/Inf."
        )

    elif candidate_blocks == 0:

        print(
            "WARNING: QK estimator was not triggered."
        )

    else:

        print(
            "QK importance hook : ACTIVE"
        )

        print(
            "QK pruning         : ACTIVE"
        )

    print("=" * 60)


except Exception as e:

    print()
    print("=" * 60)
    print("SMALL INFERENCE TEST RESULT")
    print("=" * 60)

    print(
        "Forward pass       : FAIL"
    )

    print(
        "Error type         :",
        type(e).__name__
    )

    print(
        "Error              :",
        str(e)
    )

    raise

QK ATTENTION SMALL INFERENCE TEST
Input shape: (1, 64)
Target sparsity: 90.0 %
Block size: 32x32

Running one forward pass...

SMALL INFERENCE TEST RESULT
Forward pass       : PASS
Logits shape        : (1, 64, 32000)
Logits finite       : True
NaN count           : 0
Inf count           : 0
Candidate blocks    : 4096
Kept blocks         : 1024
Threshold count     : 1024
Measured sparsity   : 75.0 %
Next token ID       : 278
Next token          : 'the'

QK importance hook : ACTIVE
QK pruning         : ACTIVE


In [ ]:
print(model.model.layers[0].self_attn.forward)
print(model.model.layers[0].self_attn.__class__)
print(model.config._attn_implementation)

<bound method _qk_block_attention_forward of LlamaAttention(
  (q_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
  (k_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
  (v_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
  (o_proj): Linear4bit(in_features=4096, out_features=4096, bias=False)
)>
<class 'transformers.models.llama.modeling_llama.LlamaAttention'>
eager
